In [ ]:
%pip -q install -U rdflib

In [ ]:
import rdflib
from rdflib import Graph, URIRef, Literal
from rdflib import Namespace
import subprocess
from pathlib import Path
import os

In [ ]:
CONFIG = {
    "Hosting": {
        "URL": "https://speakeasy.ifi.uzh.ch",
        "Username": "CyanPeekingMouse",
        "Password": "Qe5Hf3zJ"
    },
    "Data": {
        "Download_URL": "https://files.ifi.uzh.ch/ddis/teaching/2025/ATAI/dataset/",
    }
}

### Important: Before running this code

Make sure you have rclone installed on your system. You can install it:

- On macOS: `brew install rclone`
- On Linux: `sudo apt-get install rclone` or equivalent for your distribution
- On Windows: Download from rclone.org

Advantages of using rclone for this dataset:
1. Better handling of large directories (64GB in this case)
2. Resume capability if download is interrupted
3. Progress reporting
4. Efficient parallel downloads
5. Automatic retry on errors

In [ ]:
global_dir = CONFIG["Data"]["Directory"]

def download_with_rclone(url: str, output_dir: str):
    """
    Download a directory using rclone
    Args:
        url: URL of the directory to download from
        output_dir: Directory to save the downloaded files
    """
    output_path = Path(output_dir)
    global_path = Path(global_dir)
    
    # Check if data already exists
    if output_path.exists():
        print(f"Skipping download. Dataset already exists in {output_path}")
        return
    elif global_path.exists():
        print(f"Skipping download. Dataset already exists in {global_path}")
        return
    
    try:
        # Create the output directory if it doesn't exist
        output_path.mkdir(parents=True, exist_ok=True)
        print(f"Starting download to {output_path}")
        
        # Configure rclone command for HTTPS download
        cmd = ["rclone", "copy", "--progress", url, str(output_path)]
        
        # Run the command
        process = subprocess.run(cmd, check=True, text=True, capture_output=True)
        print("\nDownload completed successfully!")
        
    except subprocess.CalledProcessError as e:
        print(f"Error running rclone: {e}")
        print(f"Error output: {e.stderr}")
    except Exception as e:
        print(f"Unexpected error: {e}")

# Get configuration
url = CONFIG["Data"]["Download_URL"]
local_dir = CONFIG["Data"]["Local_Directory"]

# Download the dataset
# download_with_rclone(url, local_dir)

In [ ]:
g = Graph()
data_dir = CONFIG["Data"]["Directory"]

graph_path = os.path.join(data_dir, "graph.nt")
if os.path.exists(graph_path):
    print(f"Loading graph at {graph_path}...")
    g.parse(graph_path, format="nt")
    print(f"Loaded {graph_path} → {len(g)} triples")
else:
    print(f"Graph does not exist at {graph_path}")